In [1]:
# Installing Dependencies
%pip install lightgbm xgboost catboost scikit-learn numpy pandas scipy


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Data Load
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)

Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)


In [3]:
# 1. Session Baselines (Anchored) + Dead EDA Detection
def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

def compute_resting_baseline(sensor_df, label_df, val_col, rest_ms=180000):
    """Calculates baseline using ONLY the first 3 minutes of the session."""
    out = {}
    sess_bounds = compute_session_bounds(label_df)
    for pid, grp in sensor_df.groupby('pid'):
        t_min = sess_bounds.get(pid, (0, 0))[0]
        # Restrict to first `rest_ms`
        resting_data = grp[(grp['timestamp'] >= t_min) & (grp['timestamp'] <= t_min + rest_ms)][val_col].dropna()
        
        # Fallback to whole session if missing early data
        if len(resting_data) < 5:
            resting_data = grp[val_col].dropna()
            
        out[pid] = (resting_data.mean(), resting_data.std() + 1e-8)
    return out

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
            print(f'  {pid}: EDA zero ratio={zr:.1%} → DEAD')
    return dead

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

bl_tr_hr   = compute_resting_baseline(trainhr, train_labels, 'value')
bl_tr_eda  = compute_resting_baseline(traineda, train_labels, 'value')
bl_tr_temp = compute_resting_baseline(traintemp, train_labels, 'value')
bl_tr_ibi  = compute_resting_baseline(trainibi, train_labels, 'value')
bl_tr_acc  = compute_resting_baseline(trainacc, train_labels, 'magnitude')
bl_tr_bvp  = compute_resting_baseline(trainbvp, train_labels, 'value')

bl_te_hr   = compute_resting_baseline(testhr, test_labels, 'value')
bl_te_eda  = compute_resting_baseline(testeda, test_labels, 'value')
bl_te_temp = compute_resting_baseline(testtemp, test_labels, 'value')
bl_te_ibi  = compute_resting_baseline(testibi, test_labels, 'value')
bl_te_acc  = compute_resting_baseline(testacc, test_labels, 'magnitude')
bl_te_bvp  = compute_resting_baseline(testbvp, test_labels, 'value')

print('Train EDA dead sensors:')
TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
print('Test EDA dead sensors:')
TEST_EDA_DEAD  = eda_dead_subjects(testeda)

Train EDA dead sensors:
  70N8: EDA zero ratio=99.6% → DEAD
  Y21H: EDA zero ratio=99.4% → DEAD
Test EDA dead sensors:


In [4]:
# 2. Feature Extraction
from scipy.stats import skew as _sp_skew, kurtosis as _sp_kurt

WINDOWS_MS      = [2500, 5000, 10000]
LONG_WINDOWS_MS = [60000, 120000] # Explicit biological windows for HRV/Tonic EDA
ROLL_WIN_MS     = 30000

def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
    else:
        for s in ['mean', 'std', 'range', 'slope']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat

def ibi_extended(ibi_v, prefix, bl_m, bl_s):
    feat = {}
    n = len(ibi_v)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(ibi_v)
        feat[f'{prefix}_std']   = np.std(ibi_v)
        feat[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(ibi_v)**2))
        feat[f'{prefix}_dev']   = (np.mean(ibi_v) - bl_m) / bl_s
        diffs = np.abs(np.diff(ibi_v))
        feat[f'{prefix}_pnn50'] = np.mean(diffs > 50) if len(diffs) > 0 else np.nan
    else:
        for s in ['mean', 'std', 'rmssd', 'dev', 'pnn50']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat

def nan_eda_features(feat, wl):
    for key in list(feat.keys()):
        if f'eda_{wl}' in key and key not in [f'eda_{wl}_valid', f'eda_{wl}_zero_ratio']:
            feat[key] = np.nan
    return feat

def extract_all_features(label_df, is_train, hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp, sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid', 'timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        def get_win(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        # Short Windows (Acute changes)
        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'
            
            # HR & TEMP & ACC & BVP & EEG (Standard Logic)
            hr_v = get_win(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_hr.get(pid, (np.nan, 1))[0]) / bl_hr.get(pid, (np.nan, 1))[1] if len(hr_v) >= 1 else np.nan

            temp_v = get_win(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            
            acc_v = get_win(acc_df, 'magnitude', hw)
            if len(acc_v) >= 5:
                feat[f'acc_{wl}_mean'] = np.mean(acc_v)
                feat[f'acc_{wl}_energy'] = np.mean(acc_v**2)
            else:
                feat[f'acc_{wl}_mean'] = np.nan
                feat[f'acc_{wl}_energy'] = np.nan

            bvp_v = get_win(bvp_df, 'value', hw)
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std'] = np.std(bvp_v)
            else:
                feat[f'bvp_{wl}_std'] = np.nan

            # EDA (Phasic approximation via short standard dev)
            eda_v = get_win(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_valid'] = 0 if zr > 0.5 else 1
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, wl)
            else:
                feat[f'eda_{wl}_valid'] = np.nan
                feat = nan_eda_features(feat, wl)

        # Long Biological Windows (HRV and Tonic EDA)
        for hw in LONG_WINDOWS_MS:
            wl = f'w{hw//1000}s'
            
            # HRV strictly over long windows
            ibi_v = get_win(ibi_df, 'value', hw)
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            feat.update(ibi_extended(ibi_v, f'ibi_{wl}', bl_m, bl_s))
            
            # Tonic EDA Baseline
            eda_v_long = get_win(eda_df, 'value', hw)
            if len(eda_v_long) >= 1 and pid not in eda_dead_set:
                feat[f'eda_{wl}_tonic_mean'] = np.mean(eda_v_long)
                bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
                feat[f'eda_{wl}_tonic_dev'] = (np.mean(eda_v_long) - bl_m) / bl_s
            else:
                feat[f'eda_{wl}_tonic_mean'] = np.nan
                feat[f'eda_{wl}_tonic_dev'] = np.nan

        records.append(feat)

    return pd.DataFrame(records)

print('Feature functions ready.')

Feature functions ready.


In [5]:
# Extract TRAIN features
print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)

# Extract TEST features
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Extraction Complete.')

Extracting TRAIN features...
Extracting TEST features...
Extraction Complete.


In [6]:
# 3. Lag Features
LAG_COLS = [c for c in train_feats.columns if 'w5s' in c or 'w60s' in c]

def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid', 'timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    return df

train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id', 'pid', 'timestamp', 'arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]
print(f'Total features: {len(FEAT_COLS)}')

Total features: 115


In [7]:
# 4. Training Setup (Regression Objective)
from sklearn.metrics import balanced_accuracy_score

train_feats_sorted = train_feats.sort_values(['pid', 'timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)

# IMPORTANT: Arousal remains 1 to 5 as floats for regression
y_all  = train_feats_sorted['arousal'].values.astype(float) 
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

# Calculate sample weights to handle imbalance in regression
class_counts = pd.Series(y_all).value_counts().sort_index()
total_samples = len(y_all)
sample_weight_dict = {cls: total_samples / (len(class_counts) * count) for cls, count in class_counts.items()}
sample_weights = np.array([sample_weight_dict[y] for y in y_all])

print("Regression Setup Complete.")

Regression Setup Complete.


In [8]:
# 5. LOSO CV - Ordinal Regression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import scipy.optimize as opt

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS      = [42, 7, 123] # Reduced to 3 for speed, scale back up later

oof_blend  = np.zeros(len(train_feats_sorted))
oof_preds  = np.zeros(len(train_feats_sorted))
test_blend = np.zeros(len(test_feats))

loso_ba = []

# Optimizer for Ordinal Boundaries
def evaluate_boundaries(boundaries, preds, y_true):
    b1, b2, b3, b4 = sorted(boundaries)
    preds_binned = np.digitize(preds, bins=[b1, b2, b3, b4]) + 1
    return -balanced_accuracy_score(y_true, preds_binned)

for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr, y_tr, sw_tr = X_all[tr_mask], y_all[tr_mask], sample_weights[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    
    f_lgb = np.zeros(va_mask.sum())
    f_xgb = np.zeros(va_mask.sum())
    f_cat = np.zeros(va_mask.sum())
    t_lgb = np.zeros(len(test_feats))
    t_xgb = np.zeros(len(test_feats))
    t_cat = np.zeros(len(test_feats))

    for seed in SEEDS:
        # LGB Regression
        m_l = lgb.LGBMRegressor(
            objective='rmse', n_estimators=1000, learning_rate=0.03, 
            num_leaves=63, random_state=seed, verbose=-1
        )
        m_l.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        f_lgb += m_l.predict(X_va) / len(SEEDS)
        t_lgb += m_l.predict(X_test) / len(SEEDS)

        # XGB Regression
        m_x = xgb.XGBRegressor(
            objective='reg:squarederror', n_estimators=1000, learning_rate=0.03,
            max_depth=5, random_state=seed, verbosity=0
        )
        m_x.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
        f_xgb += m_x.predict(X_va) / len(SEEDS)
        t_xgb += m_x.predict(X_test) / len(SEEDS)
        
        # CatBoost Regression
        m_c = CatBoostRegressor(
            iterations=1000, learning_rate=0.03, depth=6, loss_function='RMSE',
            random_seed=seed, verbose=False
        )
        m_c.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)
        f_cat += m_c.predict(X_va) / len(SEEDS)
        t_cat += m_c.predict(X_test) / len(SEEDS)

    # Average the regressors
    fold_blend_preds = (f_lgb + f_xgb + f_cat) / 3
    fold_test_preds  = (t_lgb + t_xgb + t_cat) / 3
    
    oof_blend[va_mask] = fold_blend_preds
    test_blend += fold_test_preds / len(TRAIN_PIDS)
    
    # ── IN-FOLD THRESHOLD OPTIMIZATION ── (No Data Leak)
    # Optimize boundaries solely on the training data of this fold
    fold_train_preds = (m_l.predict(X_tr) + m_x.predict(X_tr) + m_c.predict(X_tr)) / 3
    initial_boundaries = [1.5, 2.5, 3.5, 4.5]
    res = opt.minimize(evaluate_boundaries, initial_boundaries, args=(fold_train_preds, y_tr), method='Nelder-Mead')
    
    optimized_boundaries = sorted(res.x)
    
    # Apply to Validation
    final_val_preds = np.digitize(fold_blend_preds, bins=optimized_boundaries) + 1
    oof_preds[va_mask] = final_val_preds
    
    ba = balanced_accuracy_score(y_va, final_val_preds)
    loso_ba.append(ba)
    print(f'  {fold_pid} — Fold BA (Opt Rounding): {ba:.4f} | Bounds: {np.round(optimized_boundaries, 2)}')

print(f'\nLOSO Mean BA: {np.mean(loso_ba):.4f} ± {np.std(loso_ba):.4f}')

  01Z2 — Fold BA (Opt Rounding): 0.3667 | Bounds: [1.64 2.63 3.42 4.15]
  70N8 — Fold BA (Opt Rounding): 0.2659 | Bounds: [1.44 2.81 3.45 4.25]
  7PF3 — Fold BA (Opt Rounding): 0.2243 | Bounds: [1.64 2.75 3.48 4.2 ]
  CQ2G — Fold BA (Opt Rounding): 0.4283 | Bounds: [1.5  2.76 3.25 4.49]
  D1XP — Fold BA (Opt Rounding): 0.0414 | Bounds: [1.44 2.76 3.26 4.48]
  DT5C — Fold BA (Opt Rounding): 0.2694 | Bounds: [1.51 2.79 3.19 4.31]
  F1ZM — Fold BA (Opt Rounding): 0.2592 | Bounds: [1.4  2.87 3.18 4.4 ]
  LIUY — Fold BA (Opt Rounding): 0.4840 | Bounds: [1.61 2.67 3.47 4.14]
  SE4Q — Fold BA (Opt Rounding): 0.1964 | Bounds: [1.76 2.6  3.43 3.89]
  TPQI — Fold BA (Opt Rounding): 0.0727 | Bounds: [1.43 2.86 3.49 4.21]
  Y21H — Fold BA (Opt Rounding): 0.1388 | Bounds: [1.49 2.73 3.23 4.5 ]

LOSO Mean BA: 0.2497 ± 0.1320


In [9]:
# 6. Final Outputs
from sklearn.metrics import classification_report

print('=== OOF Classification Report ===')
print(classification_report(y_all, oof_preds, target_names=[f'Arousal {i+1}' for i in range(5)]))

# We optimize the final test set thresholds based on the full OOF array
final_bounds = [1.5, 2.5, 3.5, 4.5]
res_final = opt.minimize(evaluate_boundaries, final_bounds, args=(oof_blend, y_all), method='Nelder-Mead')
best_final_bounds = sorted(res_final.x)

test_pred_final = np.digitize(test_blend, bins=best_final_bounds) + 1

submission = pd.DataFrame({
    'id': test_feats['id'].values,
    'arousal': test_pred_final,
})

submission.to_csv('submission-v28.csv', index=False)
print('\nsubmission-v28.csv saved.')
print(submission['arousal'].value_counts().sort_index())

=== OOF Classification Report ===
              precision    recall  f1-score   support

   Arousal 1       0.00      0.00      0.00        55
   Arousal 2       0.37      0.44      0.40       430
   Arousal 3       0.32      0.40      0.36       554
   Arousal 4       0.17      0.12      0.14       345
   Arousal 5       0.00      0.00      0.00        72

    accuracy                           0.31      1456
   macro avg       0.17      0.19      0.18      1456
weighted avg       0.27      0.31      0.29      1456


submission-v28.csv saved.
arousal
2    1036
3     433
4      27
Name: count, dtype: int64
